In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    roc_curve,
    auc,
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor
from statsmodels.tsa.arima.model import ARIMA


print("\n================================================")
print("TASK 1 : TERM DEPOSIT SUBSCRIPTION PREDICTION")
print("================================================")

# ============================================================
# LOAD DATASET
# ============================================================
df = pd.read_csv("bank-full.csv", sep=';')

# ============================================================
# EXPLORE DATASET
# ============================================================
print(df.head())
print(df.shape)
print(df.isnull().sum())
print(df['y'].value_counts())

# ============================================================
# FEATURES AND TARGET  (split BEFORE encoding to avoid leakage)
# ============================================================
X = df.drop('y', axis=1)
y = LabelEncoder().fit_transform(df['y'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ============================================================
# ENCODE CATEGORICAL FEATURES
# ============================================================
encoders = {}

for col in X_train.columns:
    if X_train[col].dtype == 'object':
        enc = LabelEncoder()
        X_train[col] = enc.fit_transform(X_train[col])
        # Handle unseen labels in test set gracefully
        X_test[col] = X_test[col].map(
            lambda x: enc.transform([x])[0] if x in enc.classes_ else -1
        )
        encoders[col] = enc

# ============================================================
# FEATURE SCALING
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ============================================================
# LOGISTIC REGRESSION  (uses scaled data)
# ============================================================
log_model = LogisticRegression(max_iter=5000)
log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)
y_prob_log = log_model.predict_proba(X_test_scaled)[:, 1]

# ============================================================
# RANDOM FOREST 
# ============================================================
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_prob_rf  = rf_model.predict_proba(X_test)[:, 1]

# ============================================================
# LOGISTIC REGRESSION EVALUATION
# ============================================================
print("\nLOGISTIC REGRESSION RESULTS")
print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred_log))
print("\nF1 SCORE")
print(f1_score(y_test, y_pred_log))
print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_pred_log))

# ============================================================
# RANDOM FOREST EVALUATION
# ============================================================
print("\nRANDOM FOREST RESULTS")
print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred_rf))
print("\nF1 SCORE")
print(f1_score(y_test, y_pred_rf))
print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_pred_rf))

# ============================================================
# ROC CURVE
# ============================================================
fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
auc_log = auc(fpr_log, tpr_log)

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_rf = auc(fpr_rf, tpr_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_log, tpr_log, label=f'Logistic Regression AUC = {auc_log:.2f}')
plt.plot(fpr_rf,  tpr_rf,  label=f'Random Forest AUC = {auc_rf:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig("roc_curve.png", dpi=150, bbox_inches='tight')  # FIX: savefig instead of show
plt.close()

# ============================================================
# SHAP EXPLAINABILITY
# ============================================================
explainer = shap.TreeExplainer(rf_model)
X_sample = X_test.iloc[:5]
shap_values = explainer.shap_values(X_sample)

# Normalise to always work with the positive-class array
if isinstance(shap_values, list):
    sv_pos = shap_values[1]           # old SHAP: list[n_classes]
else:
    sv_pos = shap_values[:, :, 1]     # new SHAP: (n_samples, n_features, n_classes)

predictions = rf_model.predict(X_sample)

print("\nSHAP PREDICTIONS")
for i in range(5):
    result = "Yes" if predictions[i] == 1 else "No"
    print(f"Customer {i+1}: {result}")

# ============================================================
# SHAP SUMMARY PLOT
# ============================================================
shap.summary_plot(sv_pos, X_sample, plot_type='bar', show=False)
plt.savefig("shap_summary.png", dpi=150, bbox_inches='tight')
plt.close()

# ============================================================
# SHAP FORCE PLOTS
# ============================================================
expected_val = (
    explainer.expected_value[1]
    if isinstance(explainer.expected_value, (list, np.ndarray))
    else explainer.expected_value
)

for i in range(5):
    shap.force_plot(
        expected_val,
        sv_pos[i],
        X_sample.iloc[i],
        matplotlib=True,
        show=False
    )
    plt.savefig(f"shap_force_customer_{i+1}.png", dpi=150, bbox_inches='tight')
    plt.close()  # FIX: prevent figure leak


# ============================================================
# TASK 2 : CUSTOMER SEGMENTATION
# ============================================================
print("\n================================================")
print("TASK 2 : CUSTOMER SEGMENTATION")
print("================================================")

mall_df = pd.read_csv("Mall_Customers.csv")

print(mall_df.head())
print(mall_df.shape)
print(mall_df.isnull().sum())
print(mall_df.info())

# ============================================================
# VISUALIZATION
# ============================================================
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', data=mall_df)
plt.title('Income vs Spending Score')
plt.savefig("income_vs_spending.png", dpi=150, bbox_inches='tight')
plt.close()

# ============================================================
# SELECT FEATURES & SCALE
# ============================================================
X_mall = mall_df[['Annual Income (k$)', 'Spending Score (1-100)']]
scaler_mall = StandardScaler()
X_scaled = scaler_mall.fit_transform(X_mall)

# ============================================================
# ELBOW METHOD
# ============================================================
inertia = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, random_state=42)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.savefig("elbow_method.png", dpi=150, bbox_inches='tight')
plt.close()

# ============================================================
# APPLY K-MEANS  
# ============================================================
kmeans = KMeans(n_clusters=5, random_state=42)
mall_df['Cluster'] = kmeans.fit_predict(X_scaled)

# ============================================================
# VISUALIZATION 
# ============================================================
mall_df['Dim1'] = X_scaled[:, 0]
mall_df['Dim2'] = X_scaled[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x='Dim1', y='Dim2',
    hue='Cluster',
    palette='Set1',
    data=mall_df,
    s=100
)
plt.xlabel('Annual Income (scaled)')
plt.ylabel('Spending Score (scaled)')
plt.title('Customer Segments')
plt.savefig("customer_segments.png", dpi=150, bbox_inches='tight')
plt.close()

# ============================================================
# MARKETING STRATEGIES
# ============================================================
print("\nMARKETING STRATEGIES")
strategies = {
    0: "Premium offers and luxury products",
    1: "Discounts and budget-friendly products",
    2: "Loyalty rewards and memberships",
    3: "Personalized advertisements",
    4: "Seasonal promotions and coupons"
}
for cluster, strategy in strategies.items():
    print(f"Cluster {cluster}: {strategy}")


# ============================================================
# TASK 3 : ENERGY CONSUMPTION FORECASTING
# ============================================================
print("\n================================================")
print("TASK 3 : ENERGY CONSUMPTION FORECASTING")
print("================================================")

power_df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',
    low_memory=False,
    na_values='?'
)

power_df['Datetime'] = pd.to_datetime(
    power_df['Date'] + ' ' + power_df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)
power_df.set_index('Datetime', inplace=True)

power_df['Global_active_power'] = pd.to_numeric(
    power_df['Global_active_power'], errors='coerce'
)
power_df = power_df.dropna()

power_hourly = power_df['Global_active_power'].resample('H').mean()

# ============================================================
# FEATURE ENGINEERING
# ============================================================
time_df = pd.DataFrame()
time_df['Power']       = power_hourly
time_df['Hour']        = time_df.index.hour
time_df['DayOfWeek']   = time_df.index.dayofweek
time_df['Month']       = time_df.index.month
time_df['Weekend']     = (time_df['DayOfWeek'] >= 5).astype(int)
time_df['Lag_1h']      = time_df['Power'].shift(1)   # FIX: 1-hour lag
time_df['Lag_24h']     = time_df['Power'].shift(24)  # FIX: 24-hour lag
time_df['Rolling_7d']  = time_df['Power'].shift(1).rolling(7 * 24).mean()  # FIX: 7-day rolling mean

time_df = time_df.dropna()

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================
train_size = int(len(time_df) * 0.8)
train = time_df.iloc[:train_size]
test  = time_df.iloc[train_size:]

# ============================================================
# ARIMA MODEL
# ============================================================
arima_series = train['Power'].iloc[-2000:]
arima_model  = ARIMA(arima_series, order=(5, 1, 0))
arima_result = arima_model.fit()
arima_forecast_raw = arima_result.forecast(steps=len(test))

arima_forecast = pd.Series(
    arima_forecast_raw.values,
    index=test.index
)

# ============================================================
# XGBOOST MODEL
# ============================================================
feature_cols = ['Hour', 'DayOfWeek', 'Month', 'Weekend', 'Lag_1h', 'Lag_24h', 'Rolling_7d']

X_train_ts = train[feature_cols]
y_train_ts = train['Power']
X_test_ts  = test[feature_cols]
y_test_ts  = test['Power']

xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_ts, y_train_ts)
xgb_forecast = xgb_model.predict(X_test_ts)

# ============================================================
# MODEL EVALUATION
# ============================================================
print("\nARIMA PERFORMANCE")
arima_mae  = mean_absolute_error(y_test_ts.values, arima_forecast.values)
arima_rmse = np.sqrt(mean_squared_error(y_test_ts.values, arima_forecast.values))
print("MAE:", arima_mae)
print("RMSE:", arima_rmse)

print("\nXGBOOST PERFORMANCE")
xgb_mae  = mean_absolute_error(y_test_ts.values, xgb_forecast)
xgb_rmse = np.sqrt(mean_squared_error(y_test_ts.values, xgb_forecast))
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)

# ============================================================
# ACTUAL VS FORECAST PLOT
# ============================================================
plt.figure(figsize=(12, 6))
plt.plot(y_test_ts.values[:200],        label='Actual')
plt.plot(arima_forecast.values[:200],   label='ARIMA Forecast')
plt.plot(xgb_forecast[:200],            label='XGBoost Forecast')
plt.title('Actual vs Forecasted Energy Usage')
plt.xlabel('Time')
plt.ylabel('Power Consumption')
plt.legend()
plt.savefig("energy_forecast.png", dpi=150, bbox_inches='tight')
plt.close()

print("\nDone. All plots saved as PNG files.")

ModuleNotFoundError: No module named 'shap'